# Evaluation: Wie gut verpixeln SAM 3, YuNet und SCRFD?

Wir vergleichen die **verpixelte Fläche** jeder Methode mit den **von Hand markierten Kopfmasken** (Ground Truth, GT) aus X-AnyLabeling.

**Zwei Maße pro Kopf bzw. Ansicht** (H = markierter Kopf, P = verpixelte Fläche):

| Maß | Formel | Frage | Ziel |
|---|---|---|---|
| Kopfabdeckung | \|P ∩ H\| / \|H\| | Wie viel vom Kopf ist verpixelt? | ≈ 100 % |
| Überverpixelung | \|P \ H\| / \|H\| | Wie viel zusätzliche Fläche, gemessen an der Kopfgröße? | klein |

Zusätzlich:
- **Abdeckung Kern:** Abdeckung gegen die um `TOLERANZ_PX` verkleinerte Kopfmaske. Fängt die Unsicherheit der eigenen Markierung am Rand ab (z. B. Kinn/Hals).
- **geschützt:** Kopfabdeckung ≥ `GESCHUETZT_AB`.
- Alles **getrennt nach Farbe (F) und S/W (SW)**, dazu beide zusammen.

**Erster Test: ein Frame (2500).** Bewertet wird hier der **Detektor mit Rand/Vergrößerung**, aber ohne Zeitlogik (hold, trail, Glätten), weil die nur über mehrere Frames wirkt.

**Datenschutz:** Das Notebook zeigt keine Bilder an. Kontrollbilder werden nur als Datei in `data/processed/` gespeichert. Vor jedem Commit die Ausgaben löschen.

## Schritt 1: Setup, Pfade und Parameter

- `VIEWS`: die beiden Ansichten wie in `SAM3.ipynb` (x0, y0, x1, y1). Ausgewertet wird **nur innerhalb** dieser Rechtecke, alles andere ist im fertigen Video schwarz.
- `GESCHUETZT_AB`: ab welcher Abdeckung ein Kopf als geschützt gilt.
- `TOLERANZ_PX`: um so viele Pixel wird die Kopfmaske für die „Kern“-Abdeckung verkleinert.

In [ ]:
import csv
import json
from pathlib import Path

import cv2
import numpy as np

PROJECT_ROOT = Path.cwd().parent                     # Notebook liegt in notebooks/
GT_DIR   = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "gt"      # Bilder + JSON aus X-AnyLabeling
EVAL_DIR = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "eval"    # Ergebnis-Tabellen (CSV)
VIS_DIR  = PROJECT_ROOT / "data" / "processed" / "PAT_1" / "eval"  # Kontrollbilder (nur lokal ansehen)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

FRAME_NR = 2500
IMG_FILE = GT_DIR / f"VID_76_f{FRAME_NR:05d}.png"
GT_FILE  = GT_DIR / f"VID_76_f{FRAME_NR:05d}.json"

H, W = 1080, 1920
VIEWS = {"F": (0, 0, 961, 1080), "SW": (991, 10, 1910, 527)}   # wie in SAM3.ipynb
VIEW_NAMES = {"F": "Farbe", "SW": "S/W"}

GESCHUETZT_AB = 0.95   # Kopf gilt als geschützt ab 95 % Abdeckung
TOLERANZ_PX   = 5      # Kopfmaske für die "Kern"-Abdeckung um 5 px verkleinern

for p in [IMG_FILE, GT_FILE]:
    print(f"{p.name}: {'gefunden' if p.exists() else 'FEHLT'}")

frame = cv2.imread(str(IMG_FILE))   # wird NICHT angezeigt
assert frame is not None and frame.shape[:2] == (H, W), "Frame fehlt oder hat falsche Größe"


def view_mask(v):
    # True innerhalb der Ansicht v
    m = np.zeros((H, W), dtype=bool)
    x0, y0, x1, y1 = VIEWS[v]
    m[y0:y1, x0:x1] = True
    return m


ANALYSED = view_mask("F") | view_mask("SW")   # alles, was ausgewertet wird

## Schritt 2: Ground Truth laden

Aus der JSON-Datei von X-AnyLabeling wird pro Polygon mit Label `head` eine Pixelmaske gezeichnet (`cv2.fillPoly`).

- **Überlappende Polygone werden zu einem Kopf vereint.** So kannst du fehlende Teile als zusätzliches Polygon ergänzen.
- **Ansicht:** Jeder Kopf wird über seinen Mittelpunkt der Ansicht F oder SW zugeordnet und auf diese Ansicht zugeschnitten (z. B. 1 px, der in den schwarzen Balken ragt).

In [ ]:
def load_gt(json_path, label="head"):
    # Liest die Polygone aus X-AnyLabeling und gibt eine Liste von Köpfen zurück
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))
    polys = []
    for s in data["shapes"]:
        if s["label"] != label or s["shape_type"] != "polygon":
            continue
        m = np.zeros((H, W), dtype=np.uint8)
        pts = np.round(np.array(s["points"])).astype(np.int32)
        cv2.fillPoly(m, [pts], 1)
        polys.append(m.astype(bool))

    # Überlappende Polygone zu einem Kopf zusammenfassen
    groups = []
    for m in polys:
        hits = [g for g in groups if (g & m).any()]
        for g in hits:
            m = m | g
        groups = [g for g in groups if not any(g is h for h in hits)] + [m]

    heads = []
    for m in groups:
        ys, xs = np.nonzero(m)
        cx, cy = xs.mean(), ys.mean()
        view = next((v for v, (x0, y0, x1, y1) in VIEWS.items() if x0 <= cx < x1 and y0 <= cy < y1), None)
        if view is None:
            print(f"WARNUNG: Kopf bei x={cx:.0f}, y={cy:.0f} liegt in keiner Ansicht -> ignoriert")
            continue
        m = m & view_mask(view)
        ys, xs = np.nonzero(m)
        heads.append({"view": view, "mask": m, "area": int(m.sum()),
                      "box": [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]})
    heads.sort(key=lambda h: (h["view"], h["box"][0]))
    for k, h in enumerate(heads):
        h["nr"] = k
    return heads, len(polys)


heads, n_polys = load_gt(GT_FILE)
print(f"{n_polys} Polygone -> {len(heads)} Köpfe")
for h in heads:
    print(f"  Kopf {h['nr']}: {VIEW_NAMES[h['view']]:5s}  Box {h['box']}  Fläche {h['area']} px")

## Schritt 3: Verpixelte Fläche jeder Methode

Jede Funktion gibt eine Maske `P` zurück (True = wird verpixelt) und eine Liste der Erkennungen. Die Parameter sind **dieselben wie in den jeweiligen Notebooks**:

| Methode | Erkennung | verpixelte Fläche |
|---|---|---|
| **YuNet** (`YuNet.ipynb`, `anonymize_video_views`) | pro Ansicht, Score ≥ 0,3 | Gesichtsbox + 50 % Rand links/rechts/unten, 80 % oben |
| **SCRFD-10G** (`SCRFD-10G.ipynb`) | ganzes Bild, 1280 × 736, Score ≥ 0,5 | Gesichtsbox + 50 % Rand an allen Seiten |
| **SAM 3** (`SAM3.ipynb`, `run_video_detect`) | pro Ansicht, Prompt „head“, Score ≥ 0,5 | Maske + 8 px; zu große Maske (> 60 % der Ansicht) unverändert („full“) |

In [ ]:
# ---------- YuNet ----------
YUNET_MODEL = PROJECT_ROOT / "models" / "face_detection_yunet_2023mar.onnx"
YUNET = {"score_threshold": 0.3, "padding": 0.5, "padding_top": 0.8}   # wie im letzten Lauf in YuNet.ipynb


def pixel_area_yunet(img, cfg=YUNET):
    # Wie anonymize_video_views in YuNet.ipynb: jede Ansicht einzeln, Rand an der Ansicht abgeschnitten
    P, dets = np.zeros((H, W), dtype=bool), []
    for v, (x0, y0, x1, y1) in VIEWS.items():
        vw, vh = x1 - x0, y1 - y0
        det = cv2.FaceDetectorYN.create(str(YUNET_MODEL), "", (vw, vh), score_threshold=cfg["score_threshold"],
                                        nms_threshold=0.3, top_k=5000)
        _, faces = det.detect(np.ascontiguousarray(img[y0:y1, x0:x1]))
        for f in ([] if faces is None else faces):
            x, y, w, h = f[:4].astype(int)
            rx1, ry1 = max(x - int(w * cfg["padding"]), 0), max(y - int(h * cfg["padding_top"]), 0)  # wie pixelate_region
            rx2, ry2 = min(x + w + int(w * cfg["padding"]), vw), min(y + h + int(h * cfg["padding"]), vh)
            P[y0 + ry1:y0 + ry2, x0 + rx1:x0 + rx2] = True
            dets.append({"view": v, "score": float(f[-1]), "mode": "ok",
                         "box": [int(x) + x0, int(y) + y0, int(x + w) + x0, int(y + h) + y0]})
    return P, dets

In [ ]:
# ---------- SCRFD-10G ----------
import onnxruntime as ort

SCRFD_MODEL = PROJECT_ROOT / "models" / "buffalo_l" / "buffalo_l" / "det_10g.onnx"
SCRFD_CFG = {"score_threshold": 0.5, "padding": 0.5}   # Start-Schwelle und Rand wie in SCRFD-10G.ipynb


class SCRFD:
    # Gekürzte Kopie aus SCRFD-10G.ipynb (Letterbox, 3 Maßstäbe, NMS)
    def __init__(self, model_path, input_size=(1280, 736), score_threshold=0.5, nms_threshold=0.4, pad_value=0):
        self.in_w, self.in_h = input_size
        self.score_threshold, self.nms_threshold, self.pad_value = score_threshold, nms_threshold, pad_value
        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]

    def detect(self, img_bgr):
        ih, iw = img_bgr.shape[:2]
        scale = min(self.in_w / iw, self.in_h / ih)
        new_w, new_h = int(iw * scale), int(ih * scale)
        canvas = np.full((self.in_h, self.in_w, 3), self.pad_value, dtype=np.uint8)
        canvas[:new_h, :new_w] = cv2.resize(img_bgr, (new_w, new_h))
        blob = cv2.dnn.blobFromImage(canvas, 1.0 / 128, (self.in_w, self.in_h), (127.5, 127.5, 127.5), swapRB=True)
        outs = self.session.run(self.output_names, {self.input_name: blob})
        boxes, scores = [], []
        for idx, stride in enumerate([8, 16, 32]):
            s = outs[idx].reshape(-1)
            d = outs[idx + 3].reshape(-1, 4) * stride
            gh, gw = self.in_h // stride, self.in_w // stride
            ys, xs = np.mgrid[:gh, :gw]
            centers = np.repeat(np.stack([xs, ys], axis=-1).astype(np.float32).reshape(-1, 2) * stride, 2, axis=0)
            for k in np.where(s >= self.score_threshold)[0]:
                cx, cy = centers[k]
                x1, y1, x2, y2 = cx - d[k, 0], cy - d[k, 1], cx + d[k, 2], cy + d[k, 3]
                boxes.append([x1 / scale, y1 / scale, (x2 - x1) / scale, (y2 - y1) / scale])
                scores.append(float(s[k]))
        if not boxes:
            return np.empty((0, 5), dtype=np.float32)
        idxs = np.array(cv2.dnn.NMSBoxes(boxes, scores, self.score_threshold, self.nms_threshold)).reshape(-1)
        return np.array([[*boxes[i], scores[i]] for i in idxs], dtype=np.float32)


def pixel_area_scrfd(img, cfg=SCRFD_CFG):
    det = SCRFD(SCRFD_MODEL, score_threshold=cfg["score_threshold"])
    P, dets = np.zeros((H, W), dtype=bool), []
    for x, y, w, h, s in det.detect(img):
        p = cfg["padding"]
        x0, x1 = int(max(0, x - p * w)), int(min(W, x + w + p * w))   # wie render_mask
        y0, y1 = int(max(0, y - p * h)), int(min(H, y + h + p * h))
        P[y0:y1, x0:x1] = True
        dets.append({"score": float(s), "box": [int(x), int(y), int(x + w), int(y + h)]})
    return P, dets


print("SCRFD-Modell:", "gefunden" if SCRFD_MODEL.exists() else f"FEHLT ({SCRFD_MODEL}) -> SCRFD wird übersprungen")

In [ ]:
# ---------- SAM 3 (braucht GPU und: uv sync --group sam3) ----------
import torch
from transformers import Sam3Model, Sam3Processor

SAM3_DIR = PROJECT_ROOT / "models" / "sam3"
SAM3_CFG = {"prompt": "head", "det_thr": 0.5, "dilate": 8, "max_frac": 0.6}   # wie run_video_detect


def pixel_area_sam3(img, cfg=SAM3_CFG):
    processor = Sam3Processor.from_pretrained(SAM3_DIR)
    model = Sam3Model.from_pretrained(SAM3_DIR, dtype=torch.bfloat16).to("cuda").eval()
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * cfg["dilate"] + 1, 2 * cfg["dilate"] + 1))
    P, dets = np.zeros((H, W), dtype=bool), []
    try:
        for v, (x0, y0, x1, y1) in VIEWS.items():
            vw, vh = x1 - x0, y1 - y0
            inputs = processor(images=cv2.cvtColor(img[y0:y1, x0:x1], cv2.COLOR_BGR2RGB), text=cfg["prompt"],
                               return_tensors="pt").to("cuda", dtype=torch.bfloat16)
            with torch.no_grad():
                out = model(**inputs)
            r = processor.post_process_instance_segmentation(
                out, threshold=cfg["det_thr"], mask_threshold=0.5, target_sizes=inputs["original_sizes"].tolist())[0]
            for m, b, s in zip(r["masks"].cpu().numpy().astype(bool), r["boxes"].cpu().tolist(),
                               r["scores"].float().cpu().tolist()):
                big = (b[2] - b[0]) > cfg["max_frac"] * vw or (b[3] - b[1]) > cfg["max_frac"] * vh
                if not big:
                    m = cv2.dilate(m.astype(np.uint8), kernel).astype(bool)
                P[y0:y1, x0:x1] |= m
                dets.append({"view": v, "score": s, "mode": "full" if big else "ok",
                             "box": [int(b[0]) + x0, int(b[1]) + y0, int(b[2]) + x0, int(b[3]) + y0]})
    finally:
        del model                      # GPU-Speicher wieder freigeben
        torch.cuda.empty_cache()
    return P, dets

## Schritt 4: Verpixelte Flächen sammeln

Zwei Ebenen werden verglichen:

1. **Detektor** (hier live berechnet): nur die Erkennung in **diesem** Frame plus Rand/Vergrößerung. Zeigt, wie gut der Detektor selbst ist.
2. **Pipeline** (aus der Datei): die Fläche, die deine **komplette Pipeline** in diesem Frame wirklich verpixelt hat, inklusive gehaltener Masken aus den Frames davor (hold). Das ist das, was im fertigen Video passiert.
   Die Datei entsteht in `SAM3.ipynb`, Abschnitt 2, Zelle „Masken für die Evaluation“ → `data/interim/PAT_1/eval/masks/VID_76_f<Frame>_<methode>.npz`.

Dauert ca. 30–60 s (vor allem das Laden von SAM 3). Ausgegeben werden nur Zahlen.

In [ ]:
# 1. Detektor allein (live auf dem PNG)
detectors = {"YuNet Detektor": pixel_area_yunet, "SAM 3 Detektor": pixel_area_sam3}
if SCRFD_MODEL.exists():
    detectors["SCRFD Detektor"] = pixel_area_scrfd

areas = {}
for name, fn in detectors.items():
    P, dets = fn(frame)
    areas[name] = P & ANALYSED          # nur innerhalb der Ansichten zählt
    print(f"{name}: {len(dets)} Erkennung(en), verpixelt {int(areas[name].sum())} px")
    for d in dets:
        print(f"    Score {d['score']:.2f}  Box {d['box']}" + (f"  {d['view']} {d['mode']}" if "view" in d else ""))

# 2. Ganze Pipeline (gespeicherte Masken)
MASK_DIR = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "eval" / "masks"
PIPELINES = {"SAM 3 Pipeline": "sam3det", "YuNet Pipeline": "yunet", "SCRFD Pipeline": "scrfd"}
for name, key in PIPELINES.items():
    f = MASK_DIR / f"VID_76_f{FRAME_NR:05d}_{key}.npz"
    if not f.exists():
        print(f"{name}: keine Maske ({f.name}) -> übersprungen")
        continue
    d = np.load(f)
    areas[name] = d["mask"] & ANALYSED
    info = {k: d[k].item() for k in d.files if k != "mask"}
    print(f"{name}: verpixelt {int(areas[name].sum())} px  | Lauf {info}")

## Schritt 5: Maße berechnen

**Pro Kopf:** Kopfabdeckung (voll und Kern) und ob der Kopf als geschützt gilt.

**Pro Ansicht** (alle Köpfe der Ansicht zusammen):
- `Abdeckung`: Anteil aller Kopfpixel, der verpixelt ist
- `Überverp.`: verpixelte Fläche außerhalb der Köpfe, geteilt durch die Kopffläche (0,5 = zusätzlich die halbe Kopffläche)
- `Präzision`: Anteil der verpixelten Fläche, der wirklich Kopf ist
- `geschützt`: geschützte Köpfe / alle Köpfe

In [ ]:
CORE_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * TOLERANZ_PX + 1, 2 * TOLERANZ_PX + 1))


def per_head(P, heads):
    rows = []
    for h in heads:
        core = cv2.erode(h["mask"].astype(np.uint8), CORE_KERNEL).astype(bool)
        if not core.any():               # sehr kleiner Kopf: Kern wäre leer -> volle Maske nehmen
            core = h["mask"]
        cov = (P & h["mask"]).sum() / h["area"]
        cov_core = (P & core).sum() / core.sum()
        rows.append({"kopf": h["nr"], "ansicht": h["view"], "flaeche_px": h["area"],
                     "abdeckung": cov, "abdeckung_kern": cov_core, "geschuetzt": cov >= GESCHUETZT_AB})
    return rows


def per_view(P, heads, head_rows):
    rows = []
    for v in ["F", "SW", "beide"]:
        region = ANALYSED if v == "beide" else view_mask(v)
        hs = [h for h in heads if v == "beide" or h["view"] == v]
        if not hs:
            continue
        Hm = np.zeros((H, W), dtype=bool)
        for h in hs:
            Hm |= h["mask"]
        Pv = P & region
        hr = [r for r in head_rows if v == "beide" or r["ansicht"] == v]
        rows.append({"ansicht": v, "koepfe": len(hs),
                     "abdeckung": (Pv & Hm).sum() / Hm.sum(),
                     "ueberverpixelung": (Pv & ~Hm).sum() / Hm.sum(),
                     "praezision": (Pv & Hm).sum() / Pv.sum() if Pv.any() else float("nan"),
                     "geschuetzt": sum(r["geschuetzt"] for r in hr)})
    return rows


head_table, view_table = [], []
for name, P in areas.items():
    hr = per_head(P, heads)
    head_table += [{"methode": name, **r} for r in hr]
    view_table += [{"methode": name, **r} for r in per_view(P, heads, hr)]

print("=== Pro Kopf ===")
print(f"{'Methode':10s} {'Kopf':>4s} {'Ansicht':7s} {'Fläche':>7s} {'Abdeckung':>9s} {'Kern':>6s}  geschützt")
for r in head_table:
    print(f"{r['methode']:10s} {r['kopf']:4d} {VIEW_NAMES[r['ansicht']]:7s} {r['flaeche_px']:7d} "
          f"{r['abdeckung']:9.1%} {r['abdeckung_kern']:6.1%}  {'ja' if r['geschuetzt'] else 'NEIN'}")

print("\n=== Pro Ansicht ===")
print(f"{'Methode':10s} {'Ansicht':7s} {'Köpfe':>5s} {'Abdeckung':>9s} {'Überverp.':>9s} {'Präzision':>9s}  geschützt")
for r in view_table:
    print(f"{r['methode']:10s} {VIEW_NAMES.get(r['ansicht'], r['ansicht']):7s} {r['koepfe']:5d} {r['abdeckung']:9.1%} "
          f"{r['ueberverpixelung']:9.2f} {r['praezision']:9.1%}  {r['geschuetzt']}/{r['koepfe']}")

## Schritt 6: Speichern

- **CSV-Tabellen** in `data/interim/PAT_1/eval/` (nur Zahlen, Grundlage für Diagramme in der Thesis)
- **Kontrollbild** pro Methode in `data/processed/PAT_1/eval/`: grün = Umriss der markierten Köpfe, rot = verpixelte Fläche. Nur **lokal** ansehen, enthält Patientendaten.

In [ ]:
def save_csv(rows, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()), delimiter=";")
        w.writeheader()
        w.writerows(rows)
    print("gespeichert:", path.relative_to(PROJECT_ROOT))


save_csv(head_table, EVAL_DIR / f"f{FRAME_NR:05d}_pro_kopf.csv")
save_csv(view_table, EVAL_DIR / f"f{FRAME_NR:05d}_pro_ansicht.csv")

for name, P in areas.items():
    vis = frame.copy()
    vis[~ANALYSED] = (vis[~ANALYSED] * 0.25).astype(np.uint8)                 # nicht ausgewertet: abgedunkelt
    vis[P] = (0.5 * vis[P] + 0.5 * np.array([0, 0, 255])).astype(np.uint8)    # rot: verpixelt
    for h in heads:
        cnts, _ = cv2.findContours(h["mask"].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(vis, cnts, -1, (0, 255, 0), 2)                        # grün: Ground Truth
    out = VIS_DIR / f"f{FRAME_NR:05d}_{name.replace(' ', '').lower()}.png"
    cv2.imwrite(str(out), vis)
    print("Kontrollbild:", out.relative_to(PROJECT_ROOT))